# Nedbank Forecast — Full Colab Pipeline
Run cells top to bottom. Runtime: T4 GPU.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Extract large parquets directly from Drive
import zipfile, os

DRIVE_DIR = '/content/drive/MyDrive/nedbank_raw'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Remove old corrupted file if it exists
corrupted = f'{DRIVE_DIR}/transactions_features.parquet'
if os.path.exists(corrupted) and os.path.getsize(corrupted) == 0:
    os.remove(corrupted)
    print('Removed corrupted transactions_features.parquet')

# Extract both zips from Drive root
for zip_name in ['transactions_features.zip', 'financials_features.zip']:
    zip_path = f'/content/drive/MyDrive/{zip_name}'
    parquet_name = zip_name.replace('.zip', '.parquet')
    
    if not os.path.exists(f'{DRIVE_DIR}/{parquet_name}'):
        print(f'Extracting {zip_name}...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DRIVE_DIR)
    else:
        print(f'{parquet_name} already exists, skipping')

print('\nFiles in nedbank_raw:', os.listdir(DRIVE_DIR))

In [ ]:
# 3. Upload small files: Train.csv, Test.csv, demographics_clean.zip
from google.colab import files
import shutil

os.makedirs('data/raw', exist_ok=True)
uploaded = files.upload()
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('data/raw')
        os.remove(fname)
    else:
        shutil.move(fname, f'data/raw/{fname}')

print(os.listdir('data/raw'))

In [ ]:
# 4. Set env vars — reads HF_TOKEN from Colab Secrets (key icon in left sidebar)
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_REPO']  = 'Zolisa/nedbank-features'
os.environ['DATA_DIR'] = DRIVE_DIR

In [ ]:
# 5. Install dependencies
!pip install -q polars==1.28.1 huggingface_hub pyarrow python-dotenv lightgbm catboost xgboost==3.2.0 joblib gdown scipy
!git clone --depth 1 https://github.com/google-research/timesfm.git /tmp/timesfm
!pip install -q -e '/tmp/timesfm[torch]'

In [ ]:
# 6. Clone repo
!git clone https://github.com/ZolisaSilolo/zindi_nedbank_forecast.git
%cd zindi_nedbank_forecast
!echo "HF_TOKEN=$HF_TOKEN" > .env
!echo "HF_REPO=$HF_REPO" >> .env
!echo "DATA_DIR=$DATA_DIR" >> .env

In [ ]:
# 7. NODE 1 — Feature engineering (reads from Drive, uploads to HF)
!python src/data/01_sagemaker_refinery.py

In [ ]:
# 8. NODE 2 — TimesFM 2.5 inference (reads from HF, writes locally)
!python src/features/02_timesfm_features.py

In [ ]:
# 9. NODE 3 — CatBoost Poisson + LightGBM 5-fold CV
!python src/models/03_hurdle_train.py

In [ ]:
# 10. NODE 4 — Generate submission
!python src/models/04_submission.py

In [ ]:
# 11. Download submission
from google.colab import files
files.download('submissions/final_submission.csv')